# Exploitation notebook (no video recording because of OS)

In [1]:
import gym
import highway_env
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from highway_env.envs.common.evaluate import PrintMetrics


# Visualisation
# import sys>
# sys.path.append('C:/Users/aless/OneDrive/Desktop/Thesis_repo_new/HighwayDRL/')     
from ipywidgets import interact, widgets
import glob

ABS_PATH = 'C:/Users/pigo/Desktop/HighwayDRL'

### Choose the environment to exploit the model in

In [2]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('dm-env-v0', 'acc-dm-env-v0', 'jam-dm-env-v0'…

<function __main__.f(input_env)>

### Choose the model

In [4]:
model_id = widgets.Text()
def m(input_model):
    model_id.value = str(input_model)
interact(m, input_model=glob.glob(f"{ABS_PATH}/training_output/models/*.zip"))


interactive(children=(Dropdown(description='input_model', options=('C:/Users/pigo/Desktop/HighwayDRL/training_…

<function __main__.m(input_model)>

In [5]:
# Number of exploitation episodes
episode_num = 3
metricObj = PrintMetrics()

env = gym.make(env_id.value)
env.configure({
    "screen_width":1500,
    "screen_height":250
})

model = PPO.load(model_id.value)

for episode in range(episode_num):

    obs, done = env.reset(), False
    decision_change_num = 0

    while not done:

        action, _ = model.predict(obs, deterministic=False)
        obs, reward, done, info = env.step(action)
        decision_change_num += env.DECISION_CHANGE

    episode_duration = (env.steps/env.config["policy_frequency"]*env.config["simulation_frequency"])/30
    decision_change_rate = decision_change_num / episode_duration
    km_travelled = (env.vehicle.speed * episode_duration)/1000
    metricObj.printEpisode(km_travelled, decision_change_num, decision_change_rate, env.vehicle.crashed, episode_duration, episode+1)

env.close()
metricObj.printRecap(episode_num)


episode 1 ended, metrics: 
	episode duration: 3 seconds, 
	km travelled: 0.06,             
	decision changes: 3 
	decision change rate: 1.0

episode 2 ended, metrics: 
	episode duration: 60 seconds, 
	km travelled: 2.16,             
	decision changes: 35 
	decision change rate: 0.58

episode 3 ended, metrics: 
	episode duration: 60 seconds, 
	km travelled: 2.16,             
	decision changes: 37 
	decision change rate: 0.62

evaluation ended, metrics:            
means: 
	mean episode duration: 41 seconds, 
	mean km travelled: 1.46, 
	mean decision changes: 25, 
	mean decision change rate: 0.73,            
total:
	total km travelled: 4.38, 
	total decision changes: 75,            
	total collision num: 1, 
	total episode num: 3
